# [실습 12-2] 프롬프트 실험 — Before/After 비교 (2순위 경로: Colab)

| 항목 | 내용 |
|---|---|
| 예상 소요 시간 | 30~40분 (**GPU 권장** — 무료 T4로 충분) |
| 본문 연계 | 12.3 프롬프트 |
| 선수 실습 | [실습 12-1] (모델 로드 감각) |
| 비용 | **완전 무료** — 유료 API 키 불요(확정 사항 ④) |

> **경로 안내**: 이 실습의 **1순위 경로는 무료 웹 챗봇**이다 — 책 지면의
> 프롬프트 쌍을 챗봇에 직접 입력해 비교하면 코드가 필요 없다(최신 무료
> 서비스 링크는 저장소 README 참고). 이 노트북은 웹 접근이 어려운 교육
> 환경을 위한 **2순위 경로**로, 같은 프롬프트 쌍을 오픈 모델
> (Qwen2.5-1.5B-Instruct, Apache-2.0)로 실행한다.

같은 과제, 다른 지시 — 프롬프트가 "업무 지시서"임을
(X)/(O) 비교로 확인한다(12.3.1 포맷 그대로).

### [준비] 모델 로드 (저장소 전용)

In [ ]:
# Colab: 런타임 유형을 T4 GPU로 설정 후 실행
# !pip -q install transformers accelerate
import platform
import torch
from transformers import (AutoTokenizer,
                          AutoModelForCausalLM)

NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForCausalLM.from_pretrained(
    NAME, torch_dtype=torch.float16,
    device_map="auto")
print("Python", platform.python_version(),
      "/ torch", torch.__version__)
print("모델 준비 완료:", NAME)

### [셀 1] 생성 함수 — ask() 래퍼 📖

In [ ]:
def ask(prompt, max_new=300):
    """프롬프트를 넣으면 응답 문자열을 돌려준다."""
    msgs = [{"role": "user", "content": prompt}]
    ids = tok.apply_chat_template(
        msgs, return_tensors="pt",
        add_generation_prompt=True).to(model.device)
    # 그리디(do_sample=False) — 두 프롬프트 비교의
    # 재현성 확보(원고 12.3 원칙, seed 불요)
    out = model.generate(ids, max_new_tokens=max_new,
                         do_sample=False)
    return tok.decode(out[0][ids.shape[-1]:],
                      skip_special_tokens=True).strip()

print(ask("한 문장으로 자기소개를 해 줘.",
          max_new=64))

**핵심 포인트**
- `apply_chat_template`가 대화 형식(역할 표시)을 모델이 배운 규격으로 감싼다 — 지시 따르기(instruction following)는 이 규격으로 미세 조정된 결과다(12.2.2).
- 그리디 디코딩(`do_sample=False`)으로 같은 질문 → 같은 답을 재현한다 — 두 프롬프트를 공정하게 비교하기 위한 통제다(12.3 재현성 원칙).

### [셀 2] (X)/(O) 프롬프트 쌍 — 12.3.1 포맷 📖

In [ ]:
MEETING = """3월 판촉 회의록입니다. 3월 5일 오후 2시, 마케팅팀 다섯 명이 참석했습니다.
이번 회의의 주제는 4월 판촉 방안입니다.
먼저 지난 분기 온라인 매출을 함께 살펴보았습니다.
매출이 목표에 조금 못 미쳤다는 점이 공유되었습니다.
특히 SNS 광고의 반응이 기대보다 낮았다는 의견이 있었습니다.
반면 신규 회원 가입은 꾸준히 늘고 있다는 이야기도 나왔습니다.
이런 배경에서 4월 광고 방식을 바꾸기로 뜻을 모았습니다.
논의 끝에, 4월 첫째 주부터 인스타그램 릴스 광고를 주 3회 올리기로 결정했습니다.
후속 작업으로, 박주임이 광고 영상 시안을 제작하기로 했습니다.
시안 완성 기한은 3월 20일입니다.
세부 예산과 문구는 추후 정하기로 했습니다."""

# (X) 모호한 지시
P_BAD = f"이거 요약해 줘.\n{MEETING}"

# (O) 역할·독자·형식을 지정 (괄호 = 개선 지점)
P_GOOD = (
    "너는 회의록 담당자다(역할).\n"
    "아래 회의록을 요약하라(과제).\n"
    "- 신입사원이 읽어도 이해되게(조건: 독자)\n"
    "- 결정 사항과 담당자를 빠뜨리지 말 것(조건)\n"
    "- '결정/할 일/기한' 3항목 불릿으로(형식)\n"
    f"{MEETING}")

**핵심 포인트**
- (O)에 붙은 괄호 주석이 12.3.1의 개선 문법이다: **역할·독자·형식** — 전작 『프롬프트를 알면 인공지능이 보인다』에서 검증된 그 원칙이다.
- 이 프롬프트 쌍은 책 지면의 것과 **동일**하다 — 1순위 경로(웹 챗봇)로 실험해도 같은 비교가 된다.

### [셀 3] Before/After 실행 📖

In [ ]:
print("═══ (X) 모호한 지시 ═══")
print(ask(P_BAD), "\n")
print("═══ (O) 역할·독자·형식 지정 ═══")
print(ask(P_GOOD))

**핵심 포인트**
- (X) 모호한 지시는 형식이 제각각인 요약을, (O) 역할·독자·형식 지정은 **결정·할 일·기한이 식별되는 구조화 요약**을 낸다 — 지정한 항목이 출력에 실제로 반영되는 것이 프롬프트의 힘이다.
- 다만 1.5B 소형 모델이라 **3항목 불릿을 문자 그대로 지키지는 못한다**(군더더기 섹션이 붙거나 항목 수가 어긋남) — 형식 이행력은 모델 크기에 좌우된다(12.2.3 스케일링의 체감판). 경로①의 웹 챗봇(대형 모델)은 훨씬 충실히 따른다.

실패 시 대처: GPU 미연결이면 [런타임 유형 변경]에서 T4 선택. 그리디라 같은 입력엔 같은 답이 재현되며, 대형 모델 경로(웹 챗봇)와의 형식 이행력 차이 자체가 관찰 포인트다(12.2.3).

### [보조 1] 기법 확장 — 퓨샷과 사고 연쇄 (12.3.2)

In [ ]:
# 퓨샷: 원하는 출력의 예시를 1~2개 먼저 보여 준다
P_FEWSHOT = (
    "고객 문의를 분류해 줘.\n"
    "예시) '환불해 주세요' → 환불\n"
    "예시) '배송이 안 와요' → 배송\n"
    "문의: '카드 결제가 두 번 됐어요' →")
print(ask(P_FEWSHOT, max_new=16), "\n")

# 사고 연쇄: 풀이 과정을 밟게 한다
P_COT = ("사과 3개들이 봉지 4개를 사고 2개를 먹었다. "
         "남은 사과는? 단계별로 계산 과정을 보이고 "
         "마지막 줄에 '답: 숫자'로 답해 줘.")
print(ask(P_COT))

### [심화 1] 만능 템플릿 빈칸 실험 (연습문제 응용 직결)

In [ ]:
# 12.3.3 만능 템플릿 — 빈칸을 채워 나만의 지시서를
# 만들고, 요소를 하나씩 빼며 품질 변화를 기록하자.
TEMPLATE = """너는 {역할}이다.
{과제}
조건: {조건}
형식: {형식}"""

my_prompt = TEMPLATE.format(
    역할="여행 잡지 에디터",
    과제="제주도 2박 3일 일정을 짜 줘.",
    조건="대중교통만 이용, 하루 예산 10만 원",
    형식="날짜별 표(아침/점심/저녁 3열)")
print(ask(my_prompt, max_new=384))

---
## 마무리

- 프롬프트는 **업무 지시서**다 — 역할·독자·조건·형식이 명확할수록 출력이 계약서처럼 돌아온다.
- (X)/(O) 비교는 어느 경로(웹 챗봇/오픈 모델)로 해도 성립한다 — 원칙은 모델을 가리지 않는다.
- 소형 모델의 한계 관찰까지가 실습이다 — 크기와 지시 이행력의 관계(12.2.3).

**연습문제 연계**: [응용 필수] "(X) 프롬프트를 원칙 명시하며 (O)로 개선" 문항은 [심화 1] 템플릿에서 바로 연습한다.

**다음 실습**: [실습 12-3] RAG 미니 체험 — 저장소 전용 (`lab-12-03_rag-mini.ipynb`)